## The more developed pickling of `pickling.ipynb`

### Creates search spaces that are used with BayesSearchCV to generate optimized model parameters

In [1]:
import pickle
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from skopt.space import Categorical, Real, Integer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
import itertools
import numpy as np
import pandas as pd
from pyhere import here

In [2]:
class RatioGenerator(BaseEstimator, TransformerMixin):
    '''
    A custom transformer that generates new features by taking the ratios of all combinations of specified columns.
    For use with the flux columns
    '''
    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y=None):
        # add a dummy attribute so sklearn knows this transformer is fitted
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Input X must be a pandas DataFrame")
        
        # Create a copy to avoid SettingWithCopy warnings or mutating the original
        X_out = X.copy()
        
        for top, bottom in itertools.combinations(self.cols, 2):
            new_col_name = f"{top}_over_{bottom}"
            X_out[new_col_name] = X_out[top] / (X_out[bottom] + 1e-8 ) #add epsilon to reduce division by 0 errors
            
        return X_out

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            raise ValueError("input_features must be provided")

        input_features = list(input_features)

        # Validate columns exist
        missing = set(self.cols) - set(input_features)
        if missing:
            raise ValueError(f"Missing columns in input_features: {missing}")

        # Generate ratio feature names
        ratio_features = [
            f"{top}_over_{bottom}"
            for top, bottom in itertools.combinations(self.cols, 2)
        ]

        # IMPORTANT: include ALL original input features
        return np.array(input_features + ratio_features, dtype=object)

#other flux cols:  'F160', 'F250', 'F350', 'F500', 'F870', 'F1100'
flux_cols = ['F8', 'F12', 'F24', 'F70']
flux_cols[::-1]

class LogRatioGenerator(BaseEstimator, TransformerMixin):
    '''
    A custom transformer that generates new features by taking the ratios of all combinations of specified columns.
    For use with the flux columns
    '''
    def __init__(self, cols):
        self.cols = cols

    def fit(self, X, y=None):
        # add a dummy attribute so sklearn knows this transformer is fitted
        self.n_features_in_ = X.shape[1]
        return self

    def transform(self, X):
        # Create a copy to avoid SettingWithCopy warnings or mutating the original
        X_out = X.copy()
        
        for top, bottom in itertools.combinations(self.cols, 2):
            new_col_name = f"log_{top}_over_{bottom}"
            X_out[new_col_name] = np.log(X_out[top] / X_out[bottom])
            
        return X_out

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            raise ValueError("input_features must be provided")

        input_features = list(input_features)

        # Validate columns exist
        missing = set(self.cols) - set(input_features)
        if missing:
            raise ValueError(f"Missing columns in input_features: {missing}")

        # Generate ratio feature names
        ratio_features = [
            f"log_{top}_over_{bottom}"
            for top, bottom in itertools.combinations(self.cols, 2)
        ]

        # IMPORTANT: include ALL original input features
        return np.array(input_features + ratio_features, dtype=object)

In [ ]:
catboost_model = CatBoostRegressor(random_state=2026, verbose=0, thread_count=-1, loss_function='RMSE')
xgboost_model = XGBRegressor(random_state=2026, verbosity=0, n_jobs=-1)
rf_model = RandomForestRegressor(random_state=2026, verbose=0, n_jobs=-1)
tree_model = DecisionTreeRegressor(random_state=2026)
svr_model = SVR(verbose=False, kernel="rbf")

model_list = [catboost_model, xgboost_model, rf_model, tree_model, svr_model]

search_space_list = [
    ## catboost
    {
        'impute': Categorical([SimpleImputer(strategy='mean'), SimpleImputer(strategy='median'), KNNImputer(), 'passthrough']),
        'ratio': Categorical(['passthrough', RatioGenerator(cols=flux_cols),LogRatioGenerator(cols=flux_cols)]),
        'scale': Categorical([StandardScaler(), RobustScaler(), 'passthrough']),
        "model__iterations": Integer(100, 3000),
        "model__learning_rate": Real(1e-4, 0.5, prior="log-uniform"),
        "model__depth": Integer(3, 12),
        "model__l2_leaf_reg": Real(1e-3, 100.0, prior="log-uniform"),
        "model__random_strength": Real(1e-9, 10.0, prior="log-uniform"),
        "model__bagging_temperature": Real(0.0, 10.0, prior="uniform"),
        "model__border_count": Integer(32, 255),
        "model__min_data_in_leaf": Integer(1, 100),
        "model__colsample_bylevel": Real(0.05, 1.0, prior="uniform"),
        "model__grow_policy": Categorical(["SymmetricTree", "Depthwise", "Lossguide"])
    },
    ## xgboost
    {
        'impute': Categorical([SimpleImputer(strategy='mean'), SimpleImputer(strategy='median'), KNNImputer(), 'passthrough']),
        'ratio': Categorical(['passthrough', RatioGenerator(cols=flux_cols),LogRatioGenerator(cols=flux_cols)]),
        'scale': Categorical([StandardScaler(), RobustScaler(), 'passthrough']),
        "model__n_estimators": Integer(100, 3000),
        "model__learning_rate": Real(1e-4, 0.5, prior="log-uniform"),
        "model__max_depth": Integer(3, 12),
        "model__min_child_weight": Integer(1, 20),
        "model__subsample": Real(0.5, 1.0, prior="uniform"),
        "model__colsample_bytree": Real(0.5, 1.0, prior="uniform"),
        "model__colsample_bylevel": Real(0.5, 1.0, prior="uniform"),
        "model__reg_alpha": Real(1e-9, 100.0, prior="log-uniform"),
        "model__reg_lambda": Real(1e-9, 100.0, prior="log-uniform"),
        "model__gamma": Real(1e-9, 10.0, prior="log-uniform")
    },
    ## random forest
    {
        'impute': Categorical([SimpleImputer(strategy='mean'), SimpleImputer(strategy='median'), KNNImputer()]),
        'ratio': Categorical(['passthrough', RatioGenerator(cols=flux_cols),LogRatioGenerator(cols=flux_cols)]),
        'scale': Categorical([StandardScaler(), RobustScaler(), 'passthrough']),
        "model__n_estimators": Integer(10, 1000),
        "model__max_depth": Integer(3, 30),
        "model__min_samples_split": Integer(2, 20),
        "model__min_samples_leaf": Integer(1, 20),
        "model__max_features": Categorical(["sqrt", "log2", None]), 
        'model__bootstrap': Categorical([True, False])
    },
    ## decision tree
    {
        'impute': Categorical([SimpleImputer(strategy='mean'), SimpleImputer(strategy='median'), KNNImputer()]),
        'ratio': Categorical(['passthrough', RatioGenerator(cols=flux_cols),LogRatioGenerator(cols=flux_cols)]),
        'scale': Categorical([StandardScaler(), RobustScaler(), 'passthrough']),
        "model__max_depth": Integer(3, 30),
        "model__min_samples_split": Integer(2, 20),
        "model__min_samples_leaf": Integer(1, 20),
        "model__max_features": Categorical(["sqrt", "log2", None])
    },
    ## SVR
    {
        'impute': Categorical([SimpleImputer(strategy='mean'), SimpleImputer(strategy='median'), KNNImputer()]),
        'ratio': Categorical(['passthrough', RatioGenerator(cols=flux_cols),LogRatioGenerator(cols=flux_cols)]),
        'scale': Categorical([StandardScaler(), RobustScaler()]),
        'model__C': Real(0.1, 100, prior='log-uniform'),
        'model__gamma': Real(1e-4, 1e+1, prior='log-uniform'),
        'model__epsilon': Real(0.01, 1.0, prior='log-uniform')
    }
]

print(len(model_list), len(search_space_list))

# with open('../../pipeline/spaces/model_list.pkl', 'wb') as f:
#     pickle.dump(model_list, f)
# with open('../../pipeline/spaces/space_list.pkl', 'wb') as f:
#     pickle.dump(search_space_list, f)

5 5
